In [ ]:
import find as fnd
import configs as cfg
import createconfig as cc
import photplotlib as pplib
import check as chk
from photplotlib import PhotometryPipeline, LightCurvePlotter

In [ ]:
# Set the directory containing the image folders and the filt
core = '/Users/nhaurber/Data/PrePolars/' # parent directory where images are and light curves will be saved
LK_PATH = core + 'lightcurves/' # path where light curves should save inside parent directory
images_location = core + 'Gaia1238/'
observer = 'NH' # initials; they will be saved in file names
DATE = '2026-06-21' # date of observation

#config_file = core + 'Gaia1238/Gaia_1238_config.ini' # path to config file for target inside parent directory

# plot phased? show comparison stars?
phased = False
comps = True

# Set the gain and read noise
RDNOISE = 1.7
GAIN = 1.76

# Set the starting aperture and annulus radii
AP_R = 8
AN_I = 22
AN_O = 27

non_sid = False

winer_loc = {'lon': -110-36/60-6.42/3600, 'lat': 31+39/60+56.08/3600, 'elevation': 1515.7}

In [ ]:
# ---- USER INPUTS FOR PHOTOMETRY ----
TARGET = "Gaia DR3 1238124314206289536 "
PERIOD = 0
IS_PHASE = False
PHASE_ZERO = 0
REF_IMAGE_NAME = '/Users/nhaurber/Data/PrePolars/Gaia1238/nha_Gaia_DR3_1238124314206289536_r_120s_2026-06-21T03-38-33_calibrated.fts.fz'
config_output = "/Users/nhaurber/Data/PrePolars/Gaia1238/"

# ---- REF STAR FINDING SETTINGS ----
N_STARS = 5
EDGE_BUFFER = 100
SAT_LIMIT = 35000
AP_R = 8
MIN_SEP_TARGET = 30

In [ ]:
### Find the target and comparison stars, and write the config file
###
### Find the target in the reference image and get its RA and Dec
src, target_ra, target_dec = cc.find_target(TARGET)
plotdata, plotwcs, nx, ny = cc.load_image(REF_IMAGE_NAME)
### This is the juicy step that finds the reference stars and writes the config file. It also queries VSX for known variable stars in the field.
### Small changes to createconfig.py can be made to change which stars are being selected as reference stars. The config file is written to the output directory specified in the config_output variable.
comp_stars = cc.find_comparison_stars(target_ra, target_dec, src, plotwcs)
config_path = cc.write_config(comp_stars, TARGET, PERIOD, IS_PHASE, PHASE_ZERO, REF_IMAGE_NAME, config_output)

# Because this is self contained, we can also read the config file back in and get the reference star coordinates for plotting.


In [ ]:
config_data = cfg.load_config(config_file)

In [ ]:
# Read in filters from headers
FILTERS = fnd.find_filts_fast(images_location)
print("Filters:")
print(FILTERS)

# Find exposure time (reads from first image)
expt = str(fnd.find_exp(images_location)) + 's'
print("Exposure time: " + expt)

TITLE = config_data['TITLE'] + "_" + expt

In [ ]:
## ---- CHECK REFERENCE IMAGE ----
## Find the RA and Dec of the target and the source image data
target_ra, target_dec, src = chk.get_target_data(config_data['TARGET_NAME'], config_data['REFERENCE_IMAGE'])
## Search for variable stars in the region and get their coordinates
vars, vsx_table = chk.variable_star_search(src, config_data['TARGET_NAME'])
## Convert the reference star pixel coordinates to RA and Dec for plotting and moving between frames
checkstars_radec = chk.stars_to_radec(config_data['STAR_COORDS'])
## Plot the reference image with the target, reference stars, and variable stars marked
chk.plot_reference_image(config_data['TARGET_NAME'], target_ra, target_dec, config_data['REFERENCE_IMAGE'], checkstars_radec, vsx_table, show_ref_stars=True, show_var_stars=True)


In [ ]:
pipe = PhotometryPipeline(
    file_dir=images_location,
    star_list_pix=config_data['STAR_COORDS'],
    ref_image=config_data['REFERENCE_IMAGE'],
    target_name=config_data['TARGET_NAME'],
    rdnoise=RDNOISE,
    aperture_radius=AP_R,
    annulus_inner=AN_I,
    annulus_outer=AN_O,
    gain=GAIN
)

# Loop over the filters and perform the photometry
for filt in FILTERS:
    print('Performing photometry for filter:', filt)
    pipe.perform_var_fwhm_phot(filt, 
                               save_ref_star_coords=False, display_apertures=False, save=True, 
                               plot=True, errorshow=True, title=config_data['TITLE'], non_sidereal=non_sid)
    pipe.calc_ap_ratio(filt, 
                       larger_ap=18, mean=True, median=False, title = config_data['TITLE'])
    
    pipe.find_mags(filt, 
                   save=True, save_ref_star_coords=False, display_apertures=False, title = config_data['TITLE'])
    
    LightCurvePlotter.plot_relative(file_dir=images_location, 
                                    filt=filt, title = config_data['TITLE'], phase=phased, comp_stars=comps, 
                                    phase_zero=config_data['PHASE_ZERO'], period=config_data['PERIOD'], save_path=LK_PATH, errorshow=True)
    
    LightCurvePlotter.plot_absolute(file_dir=images_location, 
                                    title = config_data['TITLE'], filt=filt, phase=phased, 
                                    phase_zero=config_data['PHASE_ZERO'], period=config_data['PERIOD'])